# Smart WasteVision — Waste Image Classification (TrashNet)

This notebook builds and compares image classifiers for six waste categories —
**cardboard, glass, metal, paper, plastic, trash** — using the
[TrashNet](https://github.com/garythung/trashnet) `dataset-resized` images.

**Final model:** ResNet18 pretrained on ImageNet, fine-tuned (`layer4` + classifier)
with a class-weighted loss (*Experiment 5*). The final validation and test numbers are
printed by Sections 11 and 12 when the notebook is run.

**Dataset:** 2,527 raw images → 2,524 after removing 3 exact duplicates
(stratified 70 / 15 / 15 split: 1,766 train / 379 validation / 379 test).

### How to run (fresh Colab runtime)

1. `Runtime → Change runtime type → GPU (T4)`, then `Runtime → Restart session`.
2. Run **Sections 1 → 16 top to bottom.** This is the complete, self-contained path:

| Section | Purpose |
|---|---|
| 1–2 | Imports, seed, configuration |
| 3–4 | Download TrashNet, remove duplicates, quick data analysis |
| 5–8 | Stratified split, transforms, DataLoaders, class weights |
| 9–10 | Model definition, training / evaluation utilities |
| **11** | **Experiment 5 — train the final model** (creates the checkpoint) |
| 12–14 | Final test evaluation, confusion matrix, error analysis |
| 15–16 | Save / download checkpoint, reload verification |
| 17–18 | Single-image inference, conclusion |
| Appendix A | *Optional* — Experiments 1–4 with their stored outputs (not needed to reproduce the final model) |

> The test set is used **only** to report the final model (Section 12 onward).
> It is never used to choose a model, a checkpoint or a run.

## 1. Setup & Imports

In [ ]:
# Must be set before CUDA is first used (needed for deterministic cuBLAS).
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# Core
import random
import shutil
import hashlib
import zipfile
from pathlib import Path
from collections import Counter

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from PIL import Image

# Deep learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    accuracy_score,
    precision_recall_fscore_support,
)

## 2. Configuration & Reproducibility

All constants live here and are defined once. `set_seed` seeds Python, NumPy and PyTorch;
cuDNN is put in deterministic mode. The final training run re-seeds itself in Section 11,
so its result does not depend on whether the optional Appendix experiments were run.

> Exact bit-for-bit repeatability can still differ across GPU models / library versions,
> so a re-run may land a fraction of a point away from a previous run.

In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

# Dedicated generator for the training-set shuffle (re-seeded before the final run).
train_generator = torch.Generator()
train_generator.manual_seed(SEED)

# ---------------------------------------------------------------------------
# Fixed class mapping, defined once and used everywhere in the notebook
# ---------------------------------------------------------------------------
CLASS_TO_IDX = {
    "cardboard": 0,
    "glass": 1,
    "metal": 2,
    "paper": 3,
    "plastic": 4,
    "trash": 5,
}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}
CLASSES = list(CLASS_TO_IDX.keys())
NUM_CLASSES = len(CLASSES)

# ---------------------------------------------------------------------------
# General configuration
# ---------------------------------------------------------------------------
IMG_SIZE = 224
BATCH_SIZE = 32
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Final experiment (Experiment 5: weighted fine-tuned ResNet18)
FINETUNE_EPOCHS = 8
FINETUNE_LR = 1e-4
CHECKPOINT_PATH = "best_model_resnet18_weighted_finetuned.pth"

# Custom CNN experiments (Appendix A, optional)
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: no GPU found. Training will be slow — switch the Colab runtime to GPU.")
print("Classes:", CLASS_TO_IDX)

## 3. Dataset Preparation

### 3.1 Download and extract TrashNet

In [ ]:
# Clone TrashNet (skip if already cloned — safe to re-run this cell).
if not Path("trashnet").exists():
    !git clone https://github.com/garythung/trashnet.git

zip_path = Path("trashnet/data/dataset-resized.zip")
raw_dataset_path = Path("trashnet/data/dataset-resized")

print("ZIP exists:", zip_path.exists())
print("Already extracted:", raw_dataset_path.exists())


In [ ]:
# Extract only if not already extracted.
if not raw_dataset_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(zip_path.parent)
    print("Dataset extracted successfully!")
else:
    print("Dataset already extracted — skipping.")

print("Dataset exists:", raw_dataset_path.exists())


In [ ]:
raw_class_counts = {}

for folder in sorted(raw_dataset_path.iterdir()):
    if folder.is_dir():
        raw_class_counts[folder.name] = len(list(folder.glob("*")))

print("Raw (uncleaned) class counts:")
for class_name, count in raw_class_counts.items():
    print(f"{class_name:10} -> {count} images")
print("Total:", sum(raw_class_counts.values()))


### 3.2 Duplicate detection and removal

TrashNet contains a few exact duplicate files (the same image saved more than once).
Duplicates are detected by file hash (MD5), and a **clean** copy of the dataset is
built that keeps only the first occurrence of each image. The step is programmatic,
so it stays reproducible even if the raw archive changes.

In [ ]:
image_hashes = {}     # hash -> first (kept) path
duplicate_paths = []  # paths to be dropped

for class_name in sorted(raw_class_counts.keys()):
    for image_path in sorted((raw_dataset_path / class_name).glob("*")):
        with open(image_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        if file_hash in image_hashes:
            duplicate_paths.append(image_path)
        else:
            image_hashes[file_hash] = image_path

print("Total raw images   :", sum(raw_class_counts.values()))
print("Duplicate images   :", len(duplicate_paths))
print("Unique images kept :", len(image_hashes))


In [ ]:
clean_path = Path("trashnet/clean_dataset")

if clean_path.exists():
    shutil.rmtree(clean_path)

for class_name in raw_class_counts:
    (clean_path / class_name).mkdir(parents=True, exist_ok=True)

duplicate_set = set(str(p) for p in duplicate_paths)

for class_name in sorted(raw_class_counts.keys()):
    for image_path in sorted((raw_dataset_path / class_name).glob("*")):
        if str(image_path) in duplicate_set:
            continue
        shutil.copy(image_path, clean_path / class_name / image_path.name)

clean_class_counts = {
    folder.name: len(list(folder.glob("*")))
    for folder in sorted(clean_path.iterdir()) if folder.is_dir()
}

print("Clean class counts:")
for class_name, count in clean_class_counts.items():
    print(f"{class_name:10} -> {count} images")
print("Total:", sum(clean_class_counts.values()))


## 4. Dataset Analysis

### 4.1 Class distribution

In [ ]:
data = []
for class_name in CLASSES:
    class_folder = clean_path / class_name
    for image_path in sorted(class_folder.glob("*")):
        data.append({"image_path": str(image_path), "label": class_name})

df = pd.DataFrame(data)

print("Total images:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts().reindex(CLASSES))
df.head()


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(df["label"].value_counts().reindex(CLASSES).index,
        df["label"].value_counts().reindex(CLASSES).values)
plt.xlabel("Waste Class")
plt.ylabel("Number of Images")
plt.title("Class Distribution (Cleaned Dataset)")
plt.show()


The dataset is imbalanced: **trash** has only 137 images, against 594 for **paper**.
This is why class-weighted loss is tested later in the notebook.

### 4.2 Sample images

In [ ]:
def denormalize(tensor_image):
    """Undo ImageNet normalization for display purposes."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    image = tensor_image.cpu() * std + mean
    return image.clamp(0, 1).permute(1, 2, 0).numpy()


# One random raw (un-augmented) example per class.
plt.figure(figsize=(15, 8))
for i, class_name in enumerate(CLASSES):
    image_path = random.choice(list((clean_path / class_name).glob("*")))
    image = Image.open(image_path)

    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.title(class_name)
    plt.axis("off")
plt.suptitle("One sample per class")
plt.tight_layout()
plt.show()


## 5. Train / Validation / Test Split

70% train, 15% validation, 15% test. The split is stratified on the label so every
subset keeps the original class proportions.

In [ ]:
# 70% train, 15% validation, 15% test — stratified on the label so every
# split keeps the original class proportions.
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED,
)

# Reset each split's index so `.loc[index, ...]` lookups inside the Dataset
# refer to positions within that split.
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
print("Train distribution:")
print(train_df["label"].value_counts().reindex(CLASSES))

print("\nValidation distribution:")
print(val_df["label"].value_counts().reindex(CLASSES))

print("\nTest distribution:")
print(test_df["label"].value_counts().reindex(CLASSES))


## 6. Transforms

Training images get light augmentation. Validation and test images are only resized
and normalized, using the same ImageNet statistics as training.

In [ ]:
# Training transform: light augmentation + ImageNet normalization.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Validation / test transform: deterministic resize + the same normalization
# used for training.
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## 7. Dataset & DataLoaders

In [ ]:
class WasteDataset(Dataset):
    """Loads waste images from a DataFrame with `image_path` / `label` columns."""

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_path = self.dataframe.loc[index, "image_path"]
        label_name = self.dataframe.loc[index, "label"]

        image = Image.open(image_path).convert("RGB")
        label = CLASS_TO_IDX[label_name]

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
train_dataset = WasteDataset(train_df, transform=train_transform)
val_dataset = WasteDataset(val_df, transform=val_test_transform)
test_dataset = WasteDataset(test_df, transform=val_test_transform)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

image, label = train_dataset[0]
print("\nSample image shape:", image.shape)
print("Sample label:", label, "->", IDX_TO_CLASS[label])


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,   # seeded shuffle order
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

images, labels = next(iter(train_loader))
print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)

### 7.1 Augmentation preview

In [ ]:
# A batch straight out of the training loader (shows augmentation in action).
images, labels = next(iter(train_loader))

plt.figure(figsize=(15, 8))
for i in range(min(8, len(images))):
    plt.subplot(2, 4, i + 1)
    plt.imshow(denormalize(images[i]))
    plt.title(IDX_TO_CLASS[labels[i].item()])
    plt.axis("off")
plt.suptitle("Augmented training batch")
plt.tight_layout()
plt.show()


## 8. Class Weights

Class weights are inversely proportional to class frequency and are calculated **only
from the training split**. Validation and test labels are never used.

In [ ]:
train_label_counts = train_df["label"].value_counts().reindex(CLASSES)
total_train = len(train_df)

class_weights = torch.tensor(
    [total_train / (NUM_CLASSES * train_label_counts[c]) for c in CLASSES],
    dtype=torch.float32,
).to(DEVICE)

print("Training class counts:")
print(train_label_counts)

print("\nClass weights:")
for class_name, weight in zip(CLASSES, class_weights):
    print(f"{class_name:10} -> {weight.item():.4f}")

## 9. Model Definition

One function builds the final architecture: an ImageNet-pretrained ResNet18 whose classifier
is replaced by `Dropout(0.4)` + `Linear(512, 6)`. The same function (with `pretrained=False`)
is used in Section 16 to reload the checkpoint, so training and inference architectures
cannot drift apart.

In [ ]:
def build_resnet18_classifier(num_classes=NUM_CLASSES, pretrained=True):
    """ResNet18 with the project's classifier head: Dropout(0.4) + Linear(512, num_classes)."""
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, num_classes),
    )
    return model

## 10. Training & Evaluation Utilities

**Evaluation protocol.** The best checkpoint of every experiment is chosen by
validation accuracy. The **validation set** is also used to compare experiments and
select the final model. The **test set** is reserved for reporting the selected model.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


def train_model(model, criterion, optimizer, train_loader, val_loader,
                 device, num_epochs, checkpoint_path):
    """Trains `model` and keeps the checkpoint with the best validation accuracy.

    The best checkpoint is saved to `checkpoint_path` and reloaded into the
    model before it is returned.
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            torch.save(model.state_dict(), checkpoint_path)

        flag = " (best so far -> checkpoint saved)" if improved else ""
        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"| Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}{flag}"
        )

    # Reload the best checkpoint before handing the model back.
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"\nBest validation accuracy: {best_val_acc:.4f} (loaded into model)")

    return model, history


@torch.no_grad()
def evaluate_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        predictions = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(predictions)
        all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds)

## 11. Final Experiment — Weighted Fine-Tuned ResNet18 (Experiment 5)

This is the model that is trained, saved and deployed.

| Setting | Value |
|---|---|
| Backbone | ResNet18, ImageNet-pretrained (`ResNet18_Weights.DEFAULT`) |
| Head | `Dropout(0.4)` → `Linear(512, 6)` |
| Trainable | `layer4` + `fc` only (everything else frozen) |
| Loss | `CrossEntropyLoss` with the class weights from Section 8 |
| Optimizer | Adam, learning rate `1e-4` |
| Epochs | 8 |
| Checkpoint | best **validation accuracy** → `best_model_resnet18_weighted_finetuned.pth` |

### 11.1 Build the model

In [ ]:
# Re-seed so this run does not depend on any earlier cell that consumed random numbers.
set_seed(SEED)
train_generator.manual_seed(SEED)

resnet_wt = build_resnet18_classifier(NUM_CLASSES, pretrained=True).to(DEVICE)

# Freeze everything ...
for param in resnet_wt.parameters():
    param.requires_grad = False

# ... then unfreeze layer4 and the classifier.
for param in resnet_wt.layer4.parameters():
    param.requires_grad = True
for param in resnet_wt.fc.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in resnet_wt.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in resnet_wt.parameters())
print("Trainable parameters:", trainable_params)
print("Total parameters    :", total_params)

# Weighted Cross Entropy (class_weights computed in Section 8)
criterion_wt = nn.CrossEntropyLoss(weight=class_weights)

optimizer_wt = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_wt.parameters()),
    lr=FINETUNE_LR,
)

print(resnet_wt.fc)

### 11.2 Train

In [ ]:
resnet_wt, history_resnet_wt = train_model(
    model=resnet_wt,
    criterion=criterion_wt,
    optimizer=optimizer_wt,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=FINETUNE_EPOCHS,
    checkpoint_path=CHECKPOINT_PATH,
)

### 11.3 Validation performance

In [ ]:
y_true_val_wt, y_pred_val_wt = evaluate_predictions(resnet_wt, val_loader, DEVICE)

val_acc_wt = accuracy_score(y_true_val_wt, y_pred_val_wt)
val_f1_wt = f1_score(y_true_val_wt, y_pred_val_wt, average="macro")

print(f"Weighted Fine-Tuned ResNet18 Validation Accuracy: {val_acc_wt:.4f} ({val_acc_wt * 100:.2f}%)")
print(f"Weighted Fine-Tuned ResNet18 Validation Macro F1: {val_f1_wt:.4f}")
print()

val_report_wt = classification_report(
    y_true_val_wt,
    y_pred_val_wt,
    target_names=CLASSES,
    zero_division=0,
    output_dict=True,
)
print(classification_report(
    y_true_val_wt,
    y_pred_val_wt,
    target_names=CLASSES,
    zero_division=0,
))

### 11.4 Model comparison and selection (validation data only)

Experiments 1–4 are the earlier experiments kept in **Appendix A**. Their numbers below are
copied from the stored outputs of those Appendix cells (each trained once, seed 42) and are
not recomputed here. The Experiment 5 row is computed from the run above.

Experiment 5 is the final model: it is Experiment 4 plus a class-weighted loss that was
introduced specifically to handle the small **trash** class (Section 8). Compare the
Experiment 4 and 5 rows: on a single run and a single split, differences of a point or so
are within run-to-run variation and should not be over-interpreted. **Only validation
metrics appear here — the test set is not touched until Section 12.**

In [ ]:
comparison = pd.DataFrame([
    # Experiments 1-4: copied from the stored Appendix A outputs (validation set)
    ("1. Custom CNN, standard loss",           0.6491, 0.5465, 0.00),
    ("2. Custom CNN, class-weighted loss",     0.5937, 0.5588, 0.55),
    ("3. ResNet18, frozen backbone",           0.7203, 0.6584, 0.15),
    ("4. ResNet18, fine-tuned",                0.9024, 0.8816, 0.65),
    # Experiment 5: computed from this run
    ("5. ResNet18, fine-tuned + class-weighted (this run)",
     val_acc_wt, val_f1_wt, val_report_wt["trash"]["recall"]),
], columns=["Experiment", "Val accuracy", "Val macro F1", "Trash recall (val)"])

print(comparison.to_string(
    index=False,
    formatters={
        "Val accuracy": "{:.2%}".format,
        "Val macro F1": "{:.4f}".format,
        "Trash recall (val)": "{:.2f}".format,
    },
))

## 12. Final Test Evaluation

The selected model (Experiment 5) is evaluated **once** on the held-out test set. These are
the numbers to report.

In [ ]:
# ============================================
# FINAL TEST EVALUATION — Weighted Fine-Tuned ResNet18
# ============================================
y_true_test, y_pred_test = evaluate_predictions(resnet_wt, test_loader, DEVICE)

test_acc = accuracy_score(y_true_test, y_pred_test)
test_f1 = f1_score(y_true_test, y_pred_test, average="macro")

print(f"Final Test Accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")
print(f"Final Test Macro F1: {test_f1:.4f}")
print(f"Test samples       : {len(y_true_test)}  |  misclassified: {int((y_true_test != y_pred_test).sum())}")

print("\nClassification Report:\n")
print(classification_report(
    y_true_test,
    y_pred_test,
    target_names=CLASSES,
    zero_division=0,
))

## 13. Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_true_test,
    y_pred_test
)

fig, ax = plt.subplots(figsize=(8, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASSES
)

disp.plot(
    ax=ax,
    xticks_rotation=45,
    cmap="Blues"
)

plt.title("Smart WasteVision - Final Test Confusion Matrix")
plt.tight_layout()
plt.show()

## 14. Error Analysis

### 14.1 Most common wrong predictions

In [ ]:
print("Most common wrong predictions:\n")

wrong_pairs = Counter()

for actual, predicted in zip(y_true_test, y_pred_test):

    if actual != predicted:

        pair = (
            CLASSES[actual],
            CLASSES[predicted]
        )

        wrong_pairs[pair] += 1


for (actual, predicted), count in wrong_pairs.most_common():

    print(
        f"Actual: {actual:10s} → "
        f"Predicted: {predicted:10s} = {count}"
    )

### 14.2 Misclassified test images

In [ ]:
# `test_loader` uses shuffle=False, so prediction i corresponds to row i of `test_df`.
wrong_idx = np.where(y_true_test != y_pred_test)[0]
print("Total wrong predictions:", len(wrong_idx))

# Show the first 12 wrong predictions (in test order)
plt.figure(figsize=(12, 10))

for plot_i, idx in enumerate(wrong_idx[:12]):
    image = Image.open(test_df.loc[idx, "image_path"]).convert("RGB")

    plt.subplot(3, 4, plot_i + 1)
    plt.imshow(image)
    plt.title(
        f"Actual: {CLASSES[y_true_test[idx]]}\n"
        f"Predicted: {CLASSES[y_pred_test[idx]]}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

### 14.3 Per-class summary

Computed from the test predictions above, so it always matches the current run.

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_true_test, y_pred_test, labels=list(range(NUM_CLASSES)), zero_division=0
)
per_class = pd.DataFrame({
    "precision": precision, "recall": recall, "f1": f1, "support": support
}, index=CLASSES).round(3)
print(per_class)

print("\nLowest recall   :", per_class["recall"].idxmin(), f"({per_class['recall'].min():.2f})")
print("Lowest precision:", per_class["precision"].idxmin(), f"({per_class['precision'].min():.2f})")

cm_off_diag = confusion_matrix(y_true_test, y_pred_test).copy()
np.fill_diagonal(cm_off_diag, 0)
print("\nTop 3 confusions (actual -> predicted):")
for flat in np.argsort(cm_off_diag, axis=None)[::-1][:3]:
    a, p = np.unravel_index(flat, cm_off_diag.shape)
    print(f"  {CLASSES[a]:10s} -> {CLASSES[p]:10s} : {cm_off_diag[a, p]}")

## 15. Save & Download Checkpoint

`train_model` already saved the best-validation-accuracy weights to
`best_model_resnet18_weighted_finetuned.pth` (and loaded them back into `resnet_wt`), so
this cell only checks the file and downloads it. This is the file the deployed app loads.

In [ ]:
assert os.path.exists(CHECKPOINT_PATH), "Checkpoint missing — run Section 11 first."
print("Checkpoint:", CHECKPOINT_PATH)
print("Size      :", round(os.path.getsize(CHECKPOINT_PATH) / 1024**2, 2), "MB")

try:
    from google.colab import files
    files.download(CHECKPOINT_PATH)
except ImportError:
    print("Not running in Colab — copy the checkpoint file manually.")

## 16. Verification

Rebuilds the architecture from scratch (no pretrained download), reloads the saved file with
strict key matching, and checks that it behaves exactly like the trained model. It uses the
**validation** set, so the test set stays untouched.

In [ ]:
reloaded = build_resnet18_classifier(NUM_CLASSES, pretrained=False).to(DEVICE)
state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
reloaded.load_state_dict(state)          # strict=True: raises on any key / shape mismatch
reloaded.eval()

# 1. Architecture matches the trained model
assert list(state.keys()) == list(resnet_wt.state_dict().keys()), "state_dict keys differ"
print("Classifier head :", reloaded.fc)

# 2. Reloaded model gives the same validation accuracy as the trained model
y_true_chk, y_pred_chk = evaluate_predictions(reloaded, val_loader, DEVICE)
chk_acc = accuracy_score(y_true_chk, y_pred_chk)
print(f"Validation acc  : reloaded {chk_acc:.4f} | trained {val_acc_wt:.4f}")
assert abs(chk_acc - val_acc_wt) < 1e-9, "reloaded model differs from trained model"

# 3. Class mapping is consistent (alphabetical order, same as the deployed app)
assert CLASSES == sorted(CLASSES)
assert [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)] == CLASSES
assert all(CLASS_TO_IDX[c] == i for i, c in enumerate(CLASSES))
print("Class mapping   :", CLASS_TO_IDX)
print("\nAll checks passed.")

## 17. Single-Image Inference

`predict_image` returns the predicted class and confidence for one image. It uses the
validation/test transform (resize + ImageNet normalization) — the same preprocessing as the
deployed `app.py`.

In [ ]:
def predict_image(
    model,
    image_path,
    transform,
    device,
    idx_to_class=IDX_TO_CLASS,
):
    """Predict a single waste image and return (class_name, confidence)."""
    model.eval()

    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        predicted_idx = int(torch.argmax(probabilities).item())
        confidence = float(probabilities[predicted_idx].item())

    predicted_class = idx_to_class[predicted_idx]

    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(
        f"Predicted: {predicted_class} "
        f"({confidence * 100:.1f}%)"
    )
    plt.axis("off")
    plt.show()

    print("Class probabilities:")
    for i, class_name in enumerate(CLASSES):
        print(f"{class_name:10} -> {probabilities[i].item():.4f}")

    return predicted_class, confidence

In [ ]:
# Example: predict one image from the test set with the final model.
demo_image_path = test_df.sample(1, random_state=SEED)["image_path"].iloc[0]

predicted_class, confidence = predict_image(
    resnet_wt,
    demo_image_path,
    val_test_transform,
    DEVICE,
)

print(f"\nPredicted class: {predicted_class} | Confidence: {confidence:.4f}")

## 18. Conclusion

**Result.** The class-weighted, fine-tuned ResNet18 (Experiment 5) is evaluated on 379
held-out test images in Section 12; its accuracy, macro F1, per-class metrics and confusion
matrix are reported there.

**What made the difference** (validation macro F1 from the experiments in Appendix A and Section 11):
- Transfer learning from ImageNet and fine-tuning `layer4` gave the biggest improvement
  (custom CNN 0.55 → frozen ResNet18 0.66 → fine-tuned ResNet18 0.88).
- Class weighting was added to help the small **trash** class; compare its validation
  recall in the Section 11.4 table.

**Limitations.**
- The remaining errors are mostly visually similar materials (see Section 14).
- Each configuration was trained once on a single split, and the test set is small
  (e.g. 21 trash images), so differences of a point or two should not be over-interpreted.
- The images use plain backgrounds; performance on cluttered real-world photos has not been tested.

**Skills demonstrated:** data cleaning (hash-based duplicate removal), stratified
splitting, image augmentation, custom `Dataset`/`DataLoader`, a CNN built from scratch,
transfer learning and fine-tuning, handling class imbalance, validation-based model
selection, and confusion-matrix / error analysis.

---
# Appendix A — Previous Experiments 1–4 (OPTIONAL)

**You do not need to run anything below to reproduce the final model.** Experiments 1–4 are
kept to show *why* Experiment 5 was chosen. The outputs stored in these cells come from the
earlier runs (seed 42, same split). Re-running them retrains those models and will replace
the stored outputs with slightly different numbers; the comparison table in Section 11.4
would then no longer match — so leave them unrun unless you want to reproduce the study.

They need Sections 1–10 to have been run first.

## A.1 Experiment 1 — Custom CNN (standard `CrossEntropyLoss`)

A compact CNN with four convolutional blocks (BatchNorm + ReLU + pooling),
`AdaptiveAvgPool2d`, and a dropout-regularized classifier head, trained from scratch.

In [18]:
class WasteCNN(nn.Module):
    """Compact CNN with BatchNorm + Dropout.

    Uses `AdaptiveAvgPool2d` before the classifier head so the flattened
    feature size does not depend on the input resolution.
    """

    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Quick shape sanity check.
_sample_model = WasteCNN().to(DEVICE)
images, labels = next(iter(train_loader))
outputs = _sample_model(images.to(DEVICE))
print("Input shape :", images.shape)
print("Output shape:", outputs.shape)
del _sample_model

Input shape : torch.Size([32, 3, 224, 224])
Output shape: torch.Size([32, 6])


In [19]:
model_standard = WasteCNN(num_classes=NUM_CLASSES).to(DEVICE)
criterion_standard = nn.CrossEntropyLoss()
optimizer_standard = torch.optim.Adam(model_standard.parameters(), lr=LEARNING_RATE)

model_standard, history_standard = train_model(
    model=model_standard,
    criterion=criterion_standard,
    optimizer=optimizer_standard,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    checkpoint_path="best_model_standard.pth",
)


Epoch [1/10] | Train Loss: 1.4767 | Train Acc: 0.4122 | Val Loss: 1.3483 | Val Acc: 0.4987 (best so far -> checkpoint saved)
Epoch [2/10] | Train Loss: 1.3232 | Train Acc: 0.4779 | Val Loss: 1.1922 | Val Acc: 0.5541 (best so far -> checkpoint saved)
Epoch [3/10] | Train Loss: 1.2545 | Train Acc: 0.5028 | Val Loss: 1.1407 | Val Acc: 0.5620 (best so far -> checkpoint saved)
Epoch [4/10] | Train Loss: 1.2010 | Train Acc: 0.5210 | Val Loss: 1.3495 | Val Acc: 0.5013
Epoch [5/10] | Train Loss: 1.1830 | Train Acc: 0.5476 | Val Loss: 1.0464 | Val Acc: 0.6016 (best so far -> checkpoint saved)
Epoch [6/10] | Train Loss: 1.1445 | Train Acc: 0.5646 | Val Loss: 1.3345 | Val Acc: 0.5224
Epoch [7/10] | Train Loss: 1.0986 | Train Acc: 0.5855 | Val Loss: 1.0731 | Val Acc: 0.5858
Epoch [8/10] | Train Loss: 1.0983 | Train Acc: 0.5934 | Val Loss: 1.2344 | Val Acc: 0.5541
Epoch [9/10] | Train Loss: 1.0578 | Train Acc: 0.6014 | Val Loss: 1.0555 | Val Acc: 0.6016
Epoch [10/10] | Train Loss: 1.0627 | Train Ac

### Validation performance

In [20]:
y_true_val_standard, y_pred_val_standard = evaluate_predictions(
    model_standard, val_loader, DEVICE
)

val_acc_standard = accuracy_score(y_true_val_standard, y_pred_val_standard)
val_f1_macro_standard = f1_score(
    y_true_val_standard, y_pred_val_standard, average="macro"
)

print(f"Validation Accuracy : {val_acc_standard:.4f} ({val_acc_standard * 100:.2f}%)")
print(f"Validation Macro F1  : {val_f1_macro_standard:.4f}")
print()
print(classification_report(
    y_true_val_standard,
    y_pred_val_standard,
    target_names=CLASSES,
    zero_division=0,
))

Validation Accuracy : 0.6491 (64.91%)
Validation Macro F1  : 0.5465

              precision    recall  f1-score   support

   cardboard       0.91      0.70      0.80        61
       glass       0.49      0.75      0.59        75
       metal       0.91      0.34      0.49        62
       paper       0.65      0.92      0.76        89
     plastic       0.66      0.61      0.63        72
       trash       0.00      0.00      0.00        20

    accuracy                           0.65       379
   macro avg       0.60      0.55      0.55       379
weighted avg       0.67      0.65      0.63       379



The baseline reaches 64.91% validation accuracy and 0.5465 macro F1. It never
predicts **trash** correctly (recall 0.00), the smallest class.

## A.2 Experiment 2 — Custom CNN with Class-Weighted Loss

Uses the class weights from Section 8.

In [22]:
model_weighted = WasteCNN(num_classes=NUM_CLASSES).to(DEVICE)
criterion_weighted = nn.CrossEntropyLoss(weight=class_weights)
optimizer_weighted = torch.optim.Adam(
    model_weighted.parameters(),
    lr=LEARNING_RATE,
)

model_weighted, history_weighted = train_model(
    model=model_weighted,
    criterion=criterion_weighted,
    optimizer=optimizer_weighted,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    checkpoint_path="best_model_weighted.pth",
)

Epoch [1/10] | Train Loss: 1.5518 | Train Acc: 0.3692 | Val Loss: 1.3779 | Val Acc: 0.4248 (best so far -> checkpoint saved)
Epoch [2/10] | Train Loss: 1.3969 | Train Acc: 0.4298 | Val Loss: 1.2421 | Val Acc: 0.4749 (best so far -> checkpoint saved)
Epoch [3/10] | Train Loss: 1.3558 | Train Acc: 0.4689 | Val Loss: 1.2028 | Val Acc: 0.4776 (best so far -> checkpoint saved)
Epoch [4/10] | Train Loss: 1.3244 | Train Acc: 0.4604 | Val Loss: 1.1026 | Val Acc: 0.5752 (best so far -> checkpoint saved)
Epoch [5/10] | Train Loss: 1.2532 | Train Acc: 0.5028 | Val Loss: 1.1175 | Val Acc: 0.5726
Epoch [6/10] | Train Loss: 1.1944 | Train Acc: 0.5277 | Val Loss: 1.1528 | Val Acc: 0.5937 (best so far -> checkpoint saved)
Epoch [7/10] | Train Loss: 1.2360 | Train Acc: 0.5000 | Val Loss: 1.1456 | Val Acc: 0.5462
Epoch [8/10] | Train Loss: 1.1694 | Train Acc: 0.5459 | Val Loss: 1.1377 | Val Acc: 0.5277
Epoch [9/10] | Train Loss: 1.1701 | Train Acc: 0.5447 | Val Loss: 1.0704 | Val Acc: 0.5831
Epoch [10/1

In [23]:
y_true_val_weighted, y_pred_val_weighted = evaluate_predictions(
    model_weighted, val_loader, DEVICE
)

val_acc_weighted = accuracy_score(y_true_val_weighted, y_pred_val_weighted)
val_f1_macro_weighted = f1_score(
    y_true_val_weighted, y_pred_val_weighted, average="macro"
)

print(f"Validation Accuracy : {val_acc_weighted:.4f} ({val_acc_weighted * 100:.2f}%)")
print(f"Validation Macro F1  : {val_f1_macro_weighted:.4f}")
print()
print(classification_report(
    y_true_val_weighted,
    y_pred_val_weighted,
    target_names=CLASSES,
    zero_division=0,
))

Validation Accuracy : 0.5937 (59.37%)
Validation Macro F1  : 0.5588

              precision    recall  f1-score   support

   cardboard       0.86      0.69      0.76        61
       glass       0.56      0.29      0.39        75
       metal       0.71      0.39      0.50        62
       paper       0.58      0.91      0.71        89
     plastic       0.52      0.62      0.57        72
       trash       0.34      0.55      0.42        20

    accuracy                           0.59       379
   macro avg       0.60      0.58      0.56       379
weighted avg       0.62      0.59      0.58       379



Class weighting lifts **trash** recall from 0.00 to 0.55 and macro F1 slightly
(0.5465 → 0.5588), but hurts **glass** recall (0.75 → 0.29) and overall accuracy
(64.91% → 59.37%). The custom CNN is limited by its capacity and data size,
which motivates transfer learning.

## A.3 Experiment 3 — ResNet18 Transfer Learning (frozen backbone)

A ResNet18 pretrained on ImageNet. The backbone is frozen and only the new classifier
head (`Dropout` + `Linear`) is trained.

In [ ]:
# ---------------------------------------------------------
# Experiment C: Pretrained ResNet18
# ---------------------------------------------------------

resnet = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Replace the original ImageNet classifier
in_features = resnet.fc.in_features

resnet.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, NUM_CLASSES)
)

resnet = resnet.to(DEVICE)

print(resnet.fc)

for param in resnet.parameters():
    param.requires_grad = False

for param in resnet.fc.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel()
    for p in resnet.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in resnet.parameters()
)

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

criterion_resnet = nn.CrossEntropyLoss()

optimizer_resnet = torch.optim.Adam(
    resnet.fc.parameters(),
    lr=1e-3
)

Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=512, out_features=6, bias=True)
)


Trainable parameters: 3078
Total parameters: 11179590


In [ ]:
RESNET_EPOCHS = 5

resnet, history_resnet = train_model(
    model=resnet,
    criterion=criterion_resnet,
    optimizer=optimizer_resnet,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=RESNET_EPOCHS,
    checkpoint_path="best_model_resnet18_frozen.pth",
)

Epoch [1/5] | Train Loss: 1.5386 | Train Acc: 0.3907 | Val Loss: 1.1546 | Val Acc: 0.5858 (best so far -> checkpoint saved)
Epoch [2/5] | Train Loss: 1.1488 | Train Acc: 0.5770 | Val Loss: 0.9462 | Val Acc: 0.6438 (best so far -> checkpoint saved)
Epoch [3/5] | Train Loss: 0.9851 | Train Acc: 0.6489 | Val Loss: 0.8737 | Val Acc: 0.6755 (best so far -> checkpoint saved)
Epoch [4/5] | Train Loss: 0.8964 | Train Acc: 0.6818 | Val Loss: 0.8323 | Val Acc: 0.6966 (best so far -> checkpoint saved)
Epoch [5/5] | Train Loss: 0.8836 | Train Acc: 0.6625 | Val Loss: 0.7877 | Val Acc: 0.7203 (best so far -> checkpoint saved)

Best validation accuracy: 0.7203 (loaded into model)


In [ ]:
y_true_val_resnet, y_pred_val_resnet = evaluate_predictions(
    resnet,
    val_loader,
    DEVICE
)

val_acc_resnet = accuracy_score(
    y_true_val_resnet,
    y_pred_val_resnet
)

val_f1_resnet = f1_score(
    y_true_val_resnet,
    y_pred_val_resnet,
    average="macro"
)

print(
    f"ResNet18 Validation Accuracy: "
    f"{val_acc_resnet:.4f} ({val_acc_resnet * 100:.2f}%)"
)

print(
    f"ResNet18 Validation Macro F1: "
    f"{val_f1_resnet:.4f}"
)

print()

print(
    classification_report(
        y_true_val_resnet,
        y_pred_val_resnet,
        target_names=CLASSES,
        zero_division=0
    )
)

ResNet18 Validation Accuracy: 0.7203 (72.03%)
ResNet18 Validation Macro F1: 0.6584

              precision    recall  f1-score   support

   cardboard       0.98      0.82      0.89        61
       glass       0.63      0.68      0.65        75
       metal       0.64      0.87      0.73        62
       paper       0.80      0.81      0.80        89
     plastic       0.63      0.60      0.61        72
       trash       0.75      0.15      0.25        20

    accuracy                           0.72       379
   macro avg       0.74      0.65      0.66       379
weighted avg       0.73      0.72      0.71       379



Transfer learning improves validation macro F1 from 0.5465 (custom CNN) to 0.6584,
but **trash** recall is only 0.15 — a frozen backbone is not enough here.

## A.4 Experiment 4 — ResNet18 Fine-Tuning

The last residual block (`layer4`) and the classifier are unfrozen and trained with a
lower learning rate (`1e-4`). This raises the number of trainable parameters from
3,078 to 8,396,806 (of 11,179,590).

In [ ]:
resnet_ft = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

in_features = resnet_ft.fc.in_features

resnet_ft.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, NUM_CLASSES)
)

resnet_ft = resnet_ft.to(DEVICE)

for param in resnet_ft.parameters():
    param.requires_grad = False

for param in resnet_ft.layer4.parameters():
    param.requires_grad = True

for param in resnet_ft.fc.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel()
    for p in resnet_ft.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in resnet_ft.parameters()
)

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

criterion_ft = nn.CrossEntropyLoss()

optimizer_ft = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_ft.parameters()),
    lr=1e-4
)

Trainable parameters: 8396806
Total parameters: 11179590


In [ ]:
resnet_ft, history_resnet_ft = train_model(
    model=resnet_ft,
    criterion=criterion_ft,
    optimizer=optimizer_ft,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=8,
    checkpoint_path="best_model_resnet18_finetuned.pth",
)

Epoch [1/8] | Train Loss: 1.0481 | Train Acc: 0.6116 | Val Loss: 0.6183 | Val Acc: 0.7731 (best so far -> checkpoint saved)
Epoch [2/8] | Train Loss: 0.4820 | Train Acc: 0.8539 | Val Loss: 0.4813 | Val Acc: 0.8364 (best so far -> checkpoint saved)
Epoch [3/8] | Train Loss: 0.3394 | Train Acc: 0.8901 | Val Loss: 0.3981 | Val Acc: 0.8681 (best so far -> checkpoint saved)
Epoch [4/8] | Train Loss: 0.2426 | Train Acc: 0.9275 | Val Loss: 0.3601 | Val Acc: 0.8628
Epoch [5/8] | Train Loss: 0.1998 | Train Acc: 0.9417 | Val Loss: 0.3305 | Val Acc: 0.8760 (best so far -> checkpoint saved)
Epoch [6/8] | Train Loss: 0.1464 | Train Acc: 0.9541 | Val Loss: 0.3202 | Val Acc: 0.8945 (best so far -> checkpoint saved)
Epoch [7/8] | Train Loss: 0.0889 | Train Acc: 0.9790 | Val Loss: 0.3439 | Val Acc: 0.8813
Epoch [8/8] | Train Loss: 0.0880 | Train Acc: 0.9790 | Val Loss: 0.3142 | Val Acc: 0.9024 (best so far -> checkpoint saved)

Best validation accuracy: 0.9024 (loaded into model)


In [ ]:
y_true_val_ft, y_pred_val_ft = evaluate_predictions(
    resnet_ft,
    val_loader,
    DEVICE
)

val_acc_ft = accuracy_score(
    y_true_val_ft,
    y_pred_val_ft
)

val_f1_ft = f1_score(
    y_true_val_ft,
    y_pred_val_ft,
    average="macro"
)

print(
    f"Fine-Tuned ResNet18 Validation Accuracy: "
    f"{val_acc_ft:.4f} ({val_acc_ft * 100:.2f}%)"
)

print(
    f"Fine-Tuned ResNet18 Validation Macro F1: "
    f"{val_f1_ft:.4f}"
)

print()

print(
    classification_report(
        y_true_val_ft,
        y_pred_val_ft,
        target_names=CLASSES,
        zero_division=0
    )
)

Fine-Tuned ResNet18 Validation Accuracy: 0.9024 (90.24%)
Fine-Tuned ResNet18 Validation Macro F1: 0.8816

              precision    recall  f1-score   support

   cardboard       0.98      0.90      0.94        61
       glass       0.90      0.88      0.89        75
       metal       0.87      0.94      0.90        62
       paper       0.91      0.98      0.94        89
     plastic       0.88      0.88      0.88        72
       trash       0.87      0.65      0.74        20

    accuracy                           0.90       379
   macro avg       0.90      0.87      0.88       379
weighted avg       0.90      0.90      0.90       379

